# 00 · Setup on Colab or Windows

Prepare Python and find a DSSAT executable so you can start the crop-modelling lessons.

## You will learn

- Install the course tools and understand the Colab DSSAT build.
- Use `detect()` and `connect()` to find a DSSAT executable.
- Explain the fresh `runs/` copy and resolve three common setup errors.

**Windows first:** install Python 3.10 or newer and request DSSAT 4.8 from [the DSSAT download page](https://dssat.net/downloads/), then install it in `C:/DSSAT48`.
Clone this repository, run `python -m pip install -e ".[course,dev]"` from its root during course development, and open this notebook in Jupyter or VS Code with that Python kernel.

**Colab:** open this notebook with its [Colab link](https://colab.research.google.com/github/AbdelrahmanAmr3/dssatlab/blob/master/course/00_setup/lesson.ipynb) and run cells from top to bottom.
This course is under development: the Colab path will be verified with dssatlab 0.23 before release; the saved outputs here are from Windows.

In [1]:
LESSON = "00_setup"

Load the tools, installing the course Python package first only on Colab.

In [2]:
# Load the course tools in the notebook's Python environment.
import os, shutil, subprocess, sys
from pathlib import Path
if "google.colab" in sys.modules:
    package_install = subprocess.run([sys.executable, "-m", "pip", "install", "dssatlab[course]>=0.23,<0.24"], check=True, capture_output=True)
import dssatlab as dl
import matplotlib.pyplot as plt
%matplotlib inline
print("Course tools ready; Python:", sys.version.split()[0])

Course tools ready; Python: 3.14.7


🐍 Python idea  
`sys.executable -m pip` installs into this kernel's Python; `[course]` adds pandas, matplotlib and PyYAML for tables, plots and management data.
The `if` guard skips Colab actions on Windows, and `%matplotlib inline` keeps later plots inside the notebook.

Copy the course repository once on Colab and enter this lesson folder.

In [3]:
# Open the lesson folder on Colab; local execution already starts here.
if "google.colab" in sys.modules:
    if Path.cwd().name != LESSON:
        if not Path("dssatlab").exists():
            course_clone = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/AbdelrahmanAmr3/dssatlab"], check=True, capture_output=True)
        os.chdir(Path("dssatlab") / "course" / LESSON)
print("Lesson folder:", Path.cwd().name)

Lesson folder: 00_setup


Build DSSAT on Colab; Windows uses the DSSAT 4.8 installation you prepared.

In [4]:
# Build or reuse the managed DSSAT install only on Colab.
if "google.colab" in sys.modules:
    installed_dssat = dl.install()
    print("Managed DSSAT executable:", installed_dssat.name)
else:
    print("Build skipped: use the existing Windows DSSAT installation.")

Build skipped: use the existing Windows DSSAT installation.


`install()` downloads the latest stable official DSSAT source, then uses Git, CMake and gfortran to build the executable and install its data directory.
Allow a few minutes for the first build (time varies); a usable managed install for the same version is reused.

By default the install is under `~/.cache/dssatlab/<version>` (normally `/root/.cache/dssatlab/<version>` on Colab), or `$XDG_CACHE_HOME/dssatlab/<version>` when set.
The full resolved install prefix must be at most **51 characters**; use a short, writable `XDG_CACHE_HOME` if needed.
Colab storage is temporary, so a new runtime may need another build.

Inspect discovery without installing anything, changing saved config or running DSSAT.

In [5]:
# Show all three detect fields, displaying only the executable's filename.
environment = dl.detect()
detected_dssat = environment["dssat_path"]
dl.to_dataframe([
    {"Field": "os_name", "Value": environment["os_name"], "Meaning": "Operating system"},
    {"Field": "architecture", "Value": environment["architecture"], "Meaning": "Machine architecture"},
    {"Field": "dssat_path", "Value": detected_dssat.name if detected_dssat else "None", "Meaning": "Executable found (filename only)"},
])

,Field,Value,Meaning
0,os_name,windows,Operating system
1,architecture,AMD64,Machine architecture
2,dssat_path,DSCSM048.EXE,Executable found (filename only)


`dssat_path` is a `Path` or `None`; the table shows only its filename to avoid exposing a user name.
Discovery checks saved config, `DSSAT_HOME`, platform defaults (`C:/DSSAT48` on Windows or `PATH` on Linux), then managed installs on Linux, stopping at the first valid candidate.
Finding a file does not yet prove a crop simulation will run.

Connect to the discovered DSSAT executable and remember it for later lessons.

In [6]:
# Find and remember the DSSAT executable without prompting.
dssat = dl.connect(interactive=False)
print("DSSAT executable:", dssat.name, "| File exists:", dssat.is_file())

DSSAT executable: DSCSM048.EXE | File exists: True


The shared setup cell uses `dssat = dl.connect()`; here `interactive=False` reports a missing executable immediately instead of asking for input.
An explicit `path` takes precedence: on Windows, `dl.connect(path="C:/DSSAT48")` chooses the directory directly containing the executable.
Successful connecting saves the executable path in dssatlab's config for future discovery.

Name the lesson folder `HERE` and its disposable working folder `RUNS`.

In [7]:
# Name the folders used by later lessons without creating a data copy here.
HERE = Path.cwd()
RUNS = HERE / "runs"
print("HERE:", HERE.name, "| RUNS:", RUNS.relative_to(HERE).as_posix())

HERE: 00_setup | RUNS: runs


🐍 Python idea  
`Path.cwd()` is the current folder; `/ "runs"` adds a subfolder to a `Path`.
Later lessons read inputs from `RUNS`, and print paths with `.relative_to(HERE).as_posix()` for readable forward slashes.

Preview how later setup cells replace `runs/` with fresh inputs; Lesson 00 has no data to copy.

In [8]:
# Display the shared copy commands as text without executing them here.
copy_commands = """if RUNS.exists():
    shutil.rmtree(RUNS)
_ = shutil.copytree(HERE / "data", RUNS)"""
print(copy_commands)

if RUNS.exists():
    shutil.rmtree(RUNS)
_ = shutil.copytree(HERE / "data", RUNS)


`shutil.rmtree(RUNS)` deletes the previous disposable runs and their results; `copytree` copies the committed inputs into a new `runs/`.
The final assignment suppresses a returned path, and subsequent cells read only the fresh copy.
Keep anything you want to save outside `runs/` before restarting a later lesson.

Use these three setup errors from [Troubleshooting](../../docs/guide/troubleshooting.md) to choose the next action.

In [9]:
# Match three common setup errors to their remedies.
dl.to_dataframe([
    {"Error": "DSSATNotFoundError", "Message (excerpt)": "DSSAT was not found", "Next action": "detect(); connect(path=...) or install()"},
    {"Error": "DSSATInstallError", "Message (excerpt)": "Only supported on Linux/Colab", "Next action": "Windows: install DSSAT 4.8; connect()"},
    {"Error": "DSSATInstallError", "Message (excerpt)": "Missing required tools on PATH", "Next action": "Install git, cmake, gfortran; retry install()"},
])

,Error,Message (excerpt),Next action
0,DSSATNotFoundError,DSSAT was not found,detect(); connect(path=...) or install()
1,DSSATInstallError,Only supported on Linux/Colab,Windows: install DSSAT 4.8; connect()
2,DSSATInstallError,Missing required tools on PATH,"Install git, cmake, gfortran; retry install()"


These are recovery instructions, not errors produced by this successful execution.
For missing build tools on Debian/Ubuntu, ask an administrator to install the listed tools and make them available on `PATH`, then retry `dl.install()`.

`install()` reports missing build tools rather than installing them; the unsupported-platform message still mentions `v0.1` in the current package.
See [Install and connect](../../docs/guide/install.md) for cache locations and the short-cache example.

## Your turn

Choose an automatic or explicit DSSAT location and check that connecting returns an existing file.

Hint: edit `chosen_path = None` to `chosen_path = "C:/DSSAT48"` on Windows, or `chosen_path = dl.detect()["dssat_path"]` on Colab, then run this cell.

In [10]:
# Choose a location and check it independently of the earlier discovery cells.
chosen_path = None
chosen_dssat = dl.connect(path=chosen_path, interactive=False)
dl.to_dataframe([{
    "Choice": "automatic discovery" if chosen_path is None else "explicit path",
    "DSSAT executable (filename only)": chosen_dssat.name,
    "File exists": chosen_dssat.is_file(),
}])

,Choice,DSSAT executable (filename only),File exists
0,automatic discovery,DSCSM048.EXE,True


## Recap

- Colab installs the Python tools and builds DSSAT; Windows uses an existing DSSAT 4.8 installation.
- `detect()` inspects discovery; `connect()` chooses and remembers a DSSAT executable.
- Later setup cells create fresh `RUNS` inputs, so restarting removes previous results from `runs/`.

Next: [01 · Your first simulation](../01_first_simulation/lesson.ipynb).